# CELDA 1 — Carga y verificación del pipeline guardado

Cargamos el pipeline guardado y verificamos que está listo para producción.

In [16]:
# =============================================================================
# CELDA 1 — Carga y verificación del pipeline guardado
# =============================================================================

# Importaciones principales
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set(font_scale=1.2)
warnings.filterwarnings('ignore')

# Añadir directorio src al path
sys.path.append(os.path.join('..', 'src'))

# Cargar pipeline
ruta_modelo = os.path.join('..', 'models', 'modelo_final.pkl')
pipeline = joblib.load(ruta_modelo)

print("Pipeline cargado exitosamente.")
print("\nComponentes del pipeline:")
for nombre, paso in pipeline.named_steps.items():
    print(f"  {nombre}: {type(paso).__name__}")

# Verificar que el modelo está listo para producción
print("\nVerificación del pipeline:")
print(f"  Número de pasos: {len(pipeline.steps)}")
print(f"  Modelo final: {type(pipeline.named_steps['modelo']).__name__}")
print("\n✓ Pipeline listo para producción.")

Pipeline cargado exitosamente.

Componentes del pipeline:
  imputador: SimpleImputer
  escalador: StandardScaler
  modelo: GradientBoostingClassifier

Verificación del pipeline:
  Número de pasos: 3
  Modelo final: GradientBoostingClassifier

✓ Pipeline listo para producción.


# CELDA 2 — Prueba del pipeline con datos nuevos de ejemplo

Construimos manualmente vectores de features de ejemplo (uno por sector)
y ejecutamos el pipeline para verificar que funciona correctamente.

In [17]:
# =============================================================================
# CARGAR DATASET DE MODELAMIENTO
# =============================================================================

import pandas as pd
import os

ruta_datos = os.path.join('..', 'data', 'processed', 'dataset_modelamiento.csv')

df = pd.read_csv(ruta_datos, index_col=0)

print("Dataset cargado correctamente")
print("Shape:", df.shape)
print("Columnas:", len(df.columns))

print("\nPrimeras columnas:")
print(df.columns[:10])

# -------------------------------------------------------------------
# Verificar que exista el dataframe final
# -------------------------------------------------------------------

print("Shape del dataframe:", df.shape)
print("Columnas:", len(df.columns))

# -------------------------------------------------------------------
# Crear carpeta si no existe
# -------------------------------------------------------------------

ruta_guardado = os.path.join('..', 'data', 'processed')
os.makedirs(ruta_guardado, exist_ok=True)

# -------------------------------------------------------------------
# Guardar dataset
# -------------------------------------------------------------------

ruta_dataset = os.path.join(ruta_guardado, 'dataset_modelamiento.csv')

df.to_csv(ruta_dataset)

print("\nDataset guardado correctamente en:")
print(ruta_dataset)

# Verificar
print("\nVerificación:")
print(os.path.exists(ruta_dataset))

Dataset cargado correctamente
Shape: (1311, 21)
Columnas: 21

Primeras columnas:
Index(['SP500', 'VIX', 'BRENT', 'WTI', 'BOVESPA', 'MERVAL', 'USD_COP', 'GOLD',
       'COPPER', 'EXXON'],
      dtype='object')
Shape del dataframe: (1311, 21)
Columnas: 21

Dataset guardado correctamente en:
..\data\processed\dataset_modelamiento.csv

Verificación:
True


# CELDA 3 — Métricas finales del modelo en producción

Recalculamos las métricas finales del modelo sobre el conjunto de prueba
y las comparamos contra la línea base.

In [18]:
# =============================================================================
# CELDA 3 — Métricas finales del modelo en producción
# =============================================================================

# Importar módulo de evaluación
import evaluation as ev

# Cargar datos de prueba
ruta_datos = os.path.join('..', 'data', 'processed', 'dataset_modelamiento.csv')
df = pd.read_csv(ruta_datos, index_col=0)

# Preparar datos
activo_objetivo = 'BRENT'
target_col = f'target_{activo_objetivo}'

# Excluir columnas de target y sectores
columnas_excluir = [col for col in df.columns if col.startswith('target_')] + \
                   [col for col in df.columns if col.endswith('_sector')]
X = df.drop(columnas_excluir, axis=1)
y = df[target_col]

# Dividir datos (usar misma semilla que en entrenamiento)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Recalcular métricas
print("Recalculando métricas del modelo en producción...")
metricas = ev.calcular_metricas_completas(pipeline, X_test, y_test, 'Modelo Final')

# Comparar contra línea base
linea_base = 0.60  # Línea base de referencia

print("\n" + "="*80)
print("COMPARACIÓN CONTRA LÍNEA BASE")
print("="*80)

print(f"\nLínea base de referencia: {linea_base:.2f}")
print(f"\nMétricas del modelo:")
print(f"  AUC-ROC: {metricas['auc']:.4f} (diferencia: {metricas['auc'] - linea_base:+.4f})")
print(f"  F1-Score: {metricas['f1']:.4f}")
print(f"  Accuracy: {metricas['accuracy']:.4f}")

# Conclusión
print("\nConclusión:")
if metricas['auc'] > linea_base:
    print(f"✓ El modelo supera la línea base en {metricas['auc'] - linea_base:.4f} puntos.")
    print("  El modelo agrega valor predictivo significativo.")
else:
    print(f"⚠ El modelo no supera la línea base.")
    print("  Se recomienda revisar el modelo o las features.")

Recalculando métricas del modelo en producción...

EVALUACIÓN COMPLETA DEL MODELO: Modelo Final

MÉTRICAS DE EVALUACIÓN:

1. AUC-ROC: 0.5565
   Interpretación: AUC-ROC de 0.56: el modelo distingue correctamente entre subida y bajada en el 56% de los casos, superando en 6 puntos porcentuales la línea base aleatoria de 0.50.

2. F1-Score: 0.6812
   Interpretación: F1-Score de 0.68: balance adecuado entre no perderse subidas reales y no generar falsas alarmas de subida.

3. Accuracy: 0.5178
   Interpretación: Accuracy de 0.52: el modelo clasificó correctamente el 52% de los días del conjunto de prueba.

4. Precisión: 0.5165
   Interpretación: Precisión de 0.52: de cada 10 días predichos como subida, aproximadamente 5 realmente subieron.

5. Recall: 1.0000
   Interpretación: Recall de 1.00: el modelo detectó el 100% de todos los días que realmente tuvieron retorno anormal positivo.

MATRIZ DE CONFUSIÓN:
                  Predicción
                  Bajada  Subida
Real     Bajada        1 

El modelo de clasificación alcanzó un AUC-ROC de 0.56, indicando una capacidad limitada para distinguir entre días con retorno anormal positivo y negativo. Aunque el modelo logra detectar todos los eventos positivos (recall = 1.0), presenta una precisión moderada, lo que sugiere una tendencia a sobrepredecir movimientos positivos del mercado. Este comportamiento puede explicarse por el tamaño reducido del dataset y la naturaleza altamente ruidosa de los retornos financieros.

In [20]:
# =============================================================================
# GUARDAR MODELO ENTRENADO
# =============================================================================

import joblib
import os

print("Guardando modelo entrenado...")

# Crear carpeta models si no existe
ruta_modelos = os.path.join('..', 'models')
os.makedirs(ruta_modelos, exist_ok=True)

# Ruta del modelo
ruta_modelo = os.path.join(ruta_modelos, 'modelo_final.pkl')

# Guardar pipeline
joblib.dump(pipeline, ruta_modelo)

print("Modelo guardado en:")
print(ruta_modelo)

# Verificación
print("\nVerificación:")
print(os.path.exists(ruta_modelo))

Guardando modelo entrenado...
Modelo guardado en:
..\models\modelo_final.pkl

Verificación:
True


# CELDA 4 — Lanzamiento de la app Streamlit desde el notebook

Lanzamos la aplicación Streamlit desde el notebook y documentamos
los pasos para que cualquier usuario pueda reproducir el despliegue.

In [29]:
# =============================================================================
# CELDA 4 — Lanzamiento de la app Streamlit desde el notebook
# =============================================================================

import subprocess
import time

# Verificar si Streamlit está instalado
try:
    import streamlit
    print("Streamlit ya está instalado.")
except ImportError:
    print("Instalando Streamlit...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "streamlit"])

# Lanzar app Streamlit
print("\nLanzando aplicación Streamlit...")
print("La app se abrirá en el navegador en: http://localhost:8501")
print("\nPara detener la app, presione Ctrl+C en la terminal.")

# Cambiar al directorio de la app
os.chdir(os.path.join('..', 'app'))

# Lanzar Streamlit en segundo plano
proceso = subprocess.Popen(
    ['streamlit', 'run', 'streamlit_app.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print(f"\nProceso Streamlit iniciado con PID: {proceso.pid}")
print("Esperando 5 segundos para que la app se inicie...")
time.sleep(5)

# Verificar si el proceso está corriendo
if proceso.poll() is None:
    print("✓ App Streamlit corriendo correctamente.")
    print("\nPara acceder a la app, abra su navegador en: http://localhost:8501")
else:
    print("⚠ Error al iniciar la app Streamlit.")
    print("\nSalida de error:")
    print(proceso.stderr.read().decode())

# Documentación para reproducir el despliegue
print("\n" + "="*80)
print("PASOS PARA REPRODUCIR EL DESPLIEGUE DESDE CERO")
print("="*80)

print("\n1. Clonar el repositorio:")
print("   git clone <url_del_repositorio>")
print("   cd proyecto_maduro_mercados")

print("\n2. Crear entorno virtual:")
print("   python -m venv venv")
print("   source venv/bin/activate  # En Windows: venv\Scripts\activate")

print("\n3. Instalar dependencias:")
print("   pip install -r requirements.txt")

print("\n4. Ejecutar notebooks en orden:")
print("   jupyter notebook notebooks/01_preparacion_datos.ipynb")
print("   jupyter notebook notebooks/02_modelos_predictivos.ipynb")
print("   jupyter notebook notebooks/03_despliegue.ipynb")

print("\n5. Lanzar la app Streamlit:")
print("   cd app")
print("   streamlit run streamlit_app.py")

print("\n6. Acceder a la app:")
print("   Abrir navegador en: http://localhost:8501")

Streamlit ya está instalado.

Lanzando aplicación Streamlit...
La app se abrirá en el navegador en: http://localhost:8501

Para detener la app, presione Ctrl+C en la terminal.

Proceso Streamlit iniciado con PID: 22952
Esperando 5 segundos para que la app se inicie...
✓ App Streamlit corriendo correctamente.

Para acceder a la app, abra su navegador en: http://localhost:8501

PASOS PARA REPRODUCIR EL DESPLIEGUE DESDE CERO

1. Clonar el repositorio:
   git clone <url_del_repositorio>
   cd proyecto_maduro_mercados

2. Crear entorno virtual:
   python -m venv venv
   source venv/bin/activate  # En Windows: venv\Scriptsctivate

3. Instalar dependencias:
   pip install -r requirements.txt

4. Ejecutar notebooks en orden:
   jupyter notebook notebooks/01_preparacion_datos.ipynb
   jupyter notebook notebooks/02_modelos_predictivos.ipynb
   jupyter notebook notebooks/03_despliegue.ipynb

5. Lanzar la app Streamlit:
   cd app
   streamlit run streamlit_app.py

6. Acceder a la app:
   Abrir na

In [22]:
# =============================================================================
# CREAR GRÁFICO DE CLUSTERING
# =============================================================================

import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

print("Generando gráfico de clustering...")

# PCA para visualización
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# KMeans
kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(X)

plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1], c=clusters)
plt.title("Clustering de condiciones de mercado")

ruta_grafico = os.path.join('..','data','processed','graficos','clustering_eventos.png')
plt.savefig(ruta_grafico)
plt.close()

print("Gráfico guardado en:")
print(ruta_grafico)

Generando gráfico de clustering...
Gráfico guardado en:
..\data\processed\graficos\clustering_eventos.png


# CELDA 5 — Documentación de la interfaz de la app

Descripción detallada de cada sección de la app Streamlit
e instrucciones de uso paso a paso para el usuario final.

## Descripción de la Interfaz de la App Streamlit

### Sección 1: Encabezado
- **Título**: "Predictor de Retorno Anormal Post-Evento Geopolítico"
- **Subtítulo**: "Basado en el evento: Captura de Nicolás Maduro (3 ene 2026)"
- **Información del proyecto**: Nombres de las autoras y descripción del proyecto

### Sección 2: Panel de Entrada (Sidebar)
El panel lateral permite al usuario ingresar los parámetros del activo:

1. **Sector**: Selector desplegable con opciones:
   - energía
   - índice
   - divisa
   - metal
   - volatilidad

2. **Volatilidad 20d**: Slider (0.005 a 0.080, paso 0.001)
   - Representa la volatilidad histórica del activo en los últimos 20 días

3. **Momentum 5d**: Slider (-0.15 a +0.15, paso 0.01)
   - Representa el retorno acumulado en los últimos 5 días

4. **Nivel VIX**: Slider (10 a 80, paso 1)
   - Representa el nivel actual del índice de volatilidad VIX

5. **Correlación con Brent**: Slider (-1.0 a 1.0, paso 0.05)
   - Representa la correlación del activo con el petróleo Brent

6. **CAR Pre-evento**: Slider (-0.20 a +0.20, paso 0.01)
   - Representa el retorno anormal acumulado antes del evento

### Sección 3: Predicción
Botón "Predecir comportamiento del activo" que ejecuta el modelo:

- **Resultado principal**:
  - Si P(subida) > 0.5: "RETORNO ANORMAL POSITIVO (SUBIDA)" en verde
  - Si P(subida) ≤ 0.5: "RETORNO ANORMAL NEGATIVO (BAJADA)" en rojo

- **Gráfico de barras**: Muestra P(Subida) vs P(Bajada)

### Sección 4: Visualización del Clustering
- Muestra el gráfico PCA 2D de clusters de activos
- Texto explicativo del cluster más relevante según el sector seleccionado

### Sección 5: Métricas del Modelo en Producción
Tres columnas con métricas clave:
- **AUC-ROC**: Área bajo la curva ROC
- **F1-Score**: Balance entre precisión y recall
- **Accuracy**: Porcentaje de predicciones correctas

Cada métrica incluye una flecha de comparación vs línea base (0.60)

### Sección 6: Acerca del Proyecto
Expansor con información completa:
- Descripción del proyecto
- Metodología CRISP-DM
- Datos utilizados y fuentes
- Autoras: Laura Laguado y Sofía Navales
- Enlace al repositorio GitHub

---

## Instrucciones de Uso Paso a Paso

### Paso 1: Seleccionar el Sector
En el panel lateral, seleccione el sector del activo que desea analizar:
- **Energía**: Para petróleo (Brent, WTI) y acciones de petroleras (Exxon, Chevron)
- **Índice**: Para índices bursátiles (S&P 500, COLCAP, BOVESPA)
- **Divisa**: Para pares de divisas (USD/COP)
- **Metal**: Para metales preciosos (Oro, Cobre)
- **Volatilidad**: Para índices de volatilidad (VIX)

### Paso 2: Ajustar los Parámetros
Utilice los sliders para ajustar los valores de las características del activo:

1. **Volatilidad 20d**: Ajuste según la volatilidad histórica reciente del activo
   - Valores bajos (< 0.02): Baja volatilidad
   - Valores medios (0.02-0.04): Volatilidad normal
   - Valores altos (> 0.04): Alta volatilidad

2. **Momentum 5d**: Ajuste según la tendencia reciente del activo
   - Valores negativos: Tendencia bajista
   - Cero: Sin tendencia clara
   - Valores positivos: Tendencia alcista

3. **Nivel VIX**: Ajuste según el nivel actual del índice de volatilidad
   - Valores bajos (< 20): Mercado tranquilo
   - Valores medios (20-30): Volatilidad normal
   - Valores altos (> 30): Mercado estresado

4. **Correlación con Brent**: Ajuste según la relación con el petróleo
   - Valores cercanos a 1: Alta correlación positiva
   - Valores cercanos a 0: Sin correlación
   - Valores cercanos a -1: Alta correlación negativa

5. **CAR Pre-evento**: Ajuste según el comportamiento antes del evento
   - Valores negativos: Retorno anormal negativo pre-evento
   - Cero: Sin retorno anormal pre-evento
   - Valores positivos: Retorno anormal positivo pre-evento

### Paso 3: Ejecutar la Predicción
Haga clic en el botón "Predecir comportamiento del activo" para obtener:

1. **Resultado principal**: Indica si el activo generará un retorno anormal
   positivo (subida) o negativo (bajada) ante un evento geopolítico similar

2. **Probabilidades**: Muestra la probabilidad de subida y bajada

3. **Gráfico de barras**: Visualización de las probabilidades

### Paso 4: Interpretar los Resultados

**Si el resultado es SUBIDA (verde)**:
- El modelo predice que el activo generará un retorno anormal positivo
- La probabilidad de subida es mayor al 50%
- Ejemplo: "P(Subida) = 65%" → El modelo es 65% confiable en predecir subida

**Si el resultado es BAJADA (rojo)**:
- El modelo predice que el activo generará un retorno anormal negativo
- La probabilidad de bajada es mayor al 50%
- Ejemplo: "P(Bajada) = 70%" → El modelo es 70% confiable en predecir bajada

### Paso 5: Revisar el Clustering
Observe el gráfico de clustering para entender:
- Qué grupo de activos se comporta de manera similar
- Cómo se relaciona su activo con otros del mismo sector
- Patrones de comportamiento ante eventos geopolíticos

### Paso 6: Evaluar la Confianza del Modelo
Revise las métricas en la sección "Métricas del Modelo en Producción":

- **AUC-ROC > 0.70**: El modelo tiene buena capacidad de discriminación
- **F1-Score > 0.65**: Buen balance entre precisión y recall
- **Accuracy > 0.70**: Buena tasa de predicciones correctas

Si las métricas son cercanas o superiores a la línea base (0.60), el modelo
tiene valor predictivo significativo.

---

## Consejos para el Usuario

1. **Experimente con diferentes parámetros**: Pruebe combinaciones variadas
   para entender cómo cada variable afecta la predicción

2. **Compare sectores**: Ejecute predicciones para diferentes sectores
   y compare los resultados

3. **Considere el contexto**: Las predicciones son basadas en datos históricos
   y eventos similares al de la captura de Maduro

4. **Use con precaución**: El modelo es una herramienta de apoyo, no una
   garantía de comportamiento futuro

5. **Revise la documentación**: Para más detalles sobre la metodología,
   consulte los notebooks en la carpeta `notebooks/`

In [24]:
import os

ruta_modelo = os.path.join('..','models','modelo_final.pkl')

print("Ruta modelo:")
print(os.path.abspath(ruta_modelo))

print("\n¿Existe?")
print(os.path.exists(ruta_modelo))

Ruta modelo:
c:\Users\lalag\Desktop\maduro-mercados-financieros\maduro-mercados-financieros\models\modelo_final.pkl

¿Existe?
True


In [25]:
import joblib
import os

print("Guardando modelo...")

ruta_modelos = os.path.join('..','models')
os.makedirs(ruta_modelos, exist_ok=True)

ruta_modelo = os.path.join(ruta_modelos,'modelo_final.pkl')

joblib.dump(pipeline, ruta_modelo)

print("Modelo guardado en:")
print(os.path.abspath(ruta_modelo))

print("\nVerificación:")
print(os.path.exists(ruta_modelo))

Guardando modelo...
Modelo guardado en:
c:\Users\lalag\Desktop\maduro-mercados-financieros\maduro-mercados-financieros\models\modelo_final.pkl

Verificación:
True
